# Notebook 01 — Baseline Replication

Replicates the full empirical study of **Shu et al. (2024)**: *Dynamic Asset Allocation with Asset-Specific Regime Forecasts*.

**Target outputs (paper sections)**
- Table 4 — 0/1 strategy Sharpe ratio and MDD for all 12 assets (Section 5.1)
- Table 6 — Portfolio performance comparison (Section 5.2)
- Table 7 — Correlation between forecasted and actual returns (Section 5.2)
- Figure 2 — Regime forecast plots for LargeCap, REIT, AggBond
- Figure 3 — Wealth curves for all 7 strategies

**Timeline**
- Data: 1991-01-01 → 2023-12-31
- First training window: 1991 → 2002
- Validation (λ tuning): 2002 → 2007
- Out-of-sample testing: 2007 → 2023

In [ ]:
import sys, os
from pathlib import Path

_here = Path.cwd().resolve()
_repo_root = next(
    (p for p in (_here, _here.parent, _here.parent.parent)
     if (p / "src" / "config" / "settings.py").is_file()),
    _here.parent,
)
sys.path.insert(0, str(_repo_root))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.utils.helpers import setup_logging, wealth_curve, save_results, load_results
setup_logging()

RESULTS_DIR = str(_repo_root / 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Environment ready.')

## 1 · Data Loading

In [ ]:
from src.config.settings import (
    ASSETS, ASSET_TICKERS, FRED_SERIES,
    DATA_START, DATA_END, TEST_START, TEST_END
)
from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor

loader = DataLoader()

# Download ETF prices (cached after first run)
prices = loader.load_prices(ASSET_TICKERS, start=DATA_START, end=DATA_END)
fred   = loader.load_fred(FRED_SERIES,  start=DATA_START, end=DATA_END)

print(f'Prices shape : {prices.shape}  [{prices.index[0].date()} → {prices.index[-1].date()}]')
print(f'FRED   shape : {fred.shape}')
prices.tail(3)

In [ ]:
prep = DataPreprocessor()
excess_returns, rf_daily, fred_aligned = prep.prepare(prices, fred)
total_returns = prices.pct_change().reindex(excess_returns.index).ffill()

print('Excess returns sample:')
excess_returns.describe().round(4)

## 2 · Descriptive Wealth Curves (Figure 1 equivalent)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

equity_assets = ['LargeCap', 'MidCap', 'SmallCap', 'EAFE', 'EM', 'REIT']
bond_assets   = ['AggBond', 'Treasury', 'HighYield', 'Corporate', 'Commodity', 'Gold']

for asset in equity_assets:
    wc = wealth_curve(total_returns[asset].dropna())
    axes[0].plot(wc.index, wc.values, label=asset, linewidth=1)

for asset in bond_assets:
    wc = wealth_curve(total_returns[asset].dropna())
    axes[1].plot(wc.index, wc.values, label=asset, linewidth=1)

for ax, title in zip(axes, [
    'Wealth Curves of Equity and Real Estate Indexes (1991-2023)',
    'Wealth Curves of Bond and Commodity Indexes (1991-2023)'
]):
    ax.set_yscale('log')
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=3)
    ax.grid(alpha=0.3)
    ax.set_ylabel('Wealth (log scale)')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig1_wealth_curves.png', dpi=150)
plt.show()

## 3 · Regime Identification + Forecasting (JM-XGB Framework)

Runs Algorithm 1 + Algorithm 2 for all 12 assets.  
**Expected runtime: 20-60 min** depending on hardware (biannual retraining × 32 λ candidates × 12 assets).

Results are cached to disk after the first run.

In [ ]:
import pickle
from pathlib import Path

REGIME_CACHE = Path(f'{RESULTS_DIR}/regime_forecasts.pkl')
LAMS_CACHE   = Path(f'{RESULTS_DIR}/optimal_lambdas.pkl')

if REGIME_CACHE.exists() and LAMS_CACHE.exists():
    print('Loading cached regime forecasts …')
    with open(REGIME_CACHE, 'rb') as f:
        regime_forecasts = pickle.load(f)
    with open(LAMS_CACHE, 'rb') as f:
        optimal_lambdas = pickle.load(f)
else:
    print('Running JM-XGB framework (this will take a while) …')
    from src.models.regime_framework import RegimeFramework

    framework = RegimeFramework(
        excess_returns = excess_returns,
        rf             = rf_daily,
        fred_aligned   = fred_aligned,
        assets         = ASSETS,
    )

    regime_forecasts, optimal_lambdas = framework.run(
        test_start = TEST_START,
        test_end   = TEST_END,
    )

    with open(REGIME_CACHE, 'wb') as f:
        pickle.dump(regime_forecasts, f)
    with open(LAMS_CACHE, 'wb') as f:
        pickle.dump(optimal_lambdas, f)

print(f'Regime forecasts shape: {regime_forecasts.shape}')
print(f'Bearish % per asset:')
(regime_forecasts == 1).mean().round(3) * 100

## 4 · Figure 2 — Regime Forecast Plots

In [ ]:
from src.utils.helpers import plot_wealth_curves, _shade_bear_periods

showcase_assets = ['LargeCap', 'REIT', 'AggBond']

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, asset in zip(axes, showcase_assets):
    fc   = regime_forecasts[asset].reindex(excess_returns.loc[TEST_START:TEST_END].index).ffill()
    er   = excess_returns.loc[TEST_START:TEST_END, asset]
    rf_s = rf_daily.loc[TEST_START:TEST_END]

    # 0/1 strategy wealth curve
    bull       = (1 - fc.shift(1)).fillna(0).clip(0,1)
    switches   = (fc.shift(1) != fc.shift(2)).fillna(False).astype(float)
    strat_ret  = bull * er - switches * 5e-4
    strat_ret  = strat_ret.where(bull == 1, rf_s)   # earn RF when bearish

    wc_asset  = wealth_curve(er)
    wc_strat  = wealth_curve(strat_ret)

    n_bear   = int((fc == 1).sum())
    pct_bear = (fc == 1).mean() * 100
    n_shifts = int((fc.diff().abs() > 0).sum())

    ax.plot(wc_strat.index, wc_strat.values, color='steelblue', lw=1.2,
            label=f'{asset} (0/1 strategy)')
    ax.plot(wc_asset.index, wc_asset.values, color='darkorange', lw=1.0,
            label=asset)
    _shade_bear_periods(ax, fc)
    ax.set_yscale('log')
    ax.set_title(f'{asset}: % of Bear Market: {pct_bear:.1f}%, Number of Regime Shifts: {n_shifts}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_ylabel('Wealth (log scale)')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig2_regime_forecasts.png', dpi=150)
plt.show()

## 5 · Table 4 — 0/1 Strategy Performance

In [ ]:
from src.models.regime_framework import _sharpe_01_strategy

def strategy_01_metrics(regime_fc_series, er_series, rf_series, tc=5e-4):
    """Compute Sharpe and MDD for the 0/1 strategy."""
    common = regime_fc_series.index.intersection(er_series.index)
    fc  = regime_fc_series.reindex(common)
    er  = er_series.reindex(common)
    rf_ = rf_series.reindex(common).fillna(0)

    bull      = (1 - fc.shift(1)).fillna(0).clip(0,1)
    switches  = (fc.shift(1) != fc.shift(2)).fillna(False).astype(float)
    strat_ret = bull * er - switches * tc

    ann_mean  = strat_ret.mean() * 252
    ann_std   = strat_ret.std()  * np.sqrt(252)
    sharpe    = ann_mean / ann_std if ann_std > 1e-10 else 0.0

    cum  = (1 + strat_ret).cumprod()
    peak = cum.cummax()
    mdd  = float(((cum - peak) / peak).min())

    return {'Sharpe': round(sharpe, 2), 'MDD': f"{mdd:.2%}"}


er_test = excess_returns.loc[TEST_START:TEST_END]
rf_test = rf_daily.loc[TEST_START:TEST_END]

# Buy-and-hold (all-bullish dummy)
bnh_dummy = pd.Series(0, index=er_test.index)

rows_sr, rows_mdd = {}, {}
for asset in ASSETS:
    er_a = er_test[asset]
    bnh  = strategy_01_metrics(bnh_dummy, er_a, rf_test)
    jmx  = strategy_01_metrics(regime_forecasts[asset], er_a, rf_test)

    rows_sr[asset]  = {'B&H': bnh['Sharpe'], 'JM-XGB': jmx['Sharpe']}
    rows_mdd[asset] = {'B&H': bnh['MDD'],    'JM-XGB': jmx['MDD']}

table4_sr  = pd.DataFrame(rows_sr).T
table4_mdd = pd.DataFrame(rows_mdd).T

print('=== Table 4 — Sharpe Ratio ===')
print(table4_sr.to_string())
print()
print('=== Table 4 — Maximum Drawdown ===')
print(table4_mdd.to_string())

table4_sr.to_csv(f'{RESULTS_DIR}/table4_sharpe.csv')
table4_mdd.to_csv(f'{RESULTS_DIR}/table4_mdd.csv')

## 6 · Regime-Conditional Return Forecasts (for MV-JM-XGB)

In [ ]:
# For MV(JM-XGB), the return forecast on day t is:
# mean excess return of all training-window days classified in the same regime
# as forecasted for t+1, using the JM with the optimal lambda.

# Here we approximate with a simplified rolling version:
# for each day, use the last 11-year window's regime-conditional mean return.
from src.config.settings import TRAIN_YEARS, MV_BEAR_CAP

RCRET_CACHE = Path(f'{RESULTS_DIR}/regime_cond_returns.pkl')

if RCRET_CACHE.exists():
    with open(RCRET_CACHE, 'rb') as f:
        regime_cond_returns = pickle.load(f)
else:
    regime_cond_returns_dict = {}
    for asset in ASSETS:
        rc = pd.Series(index=er_test.index, dtype=float)
        for dt in er_test.index:
            start_w = dt - pd.DateOffset(years=TRAIN_YEARS)
            hist_er = excess_returns.loc[start_w:dt, asset]
            hist_fc = regime_forecasts[asset].reindex(hist_er.index).ffill()
            _pr = regime_forecasts[asset].reindex([dt]).squeeze()
            pred = int(_pr) if pd.notna(_pr) else 1
            mask    = (hist_fc == pred)
            if mask.sum() > 5:
                mu = float(hist_er[mask].mean())
            else:
                mu = float(hist_er.mean())
            # cap bearish
            if pred == 1:
                mu = max(mu, MV_BEAR_CAP)
            rc[dt] = mu
        regime_cond_returns_dict[asset] = rc

    regime_cond_returns = pd.DataFrame(regime_cond_returns_dict)
    with open(RCRET_CACHE, 'wb') as f:
        pickle.dump(regime_cond_returns, f)

print('Regime-conditional return forecasts computed.')
regime_cond_returns.describe().round(5)

## 7 · Table 7 — Forecast Correlation

In [ ]:
from src.config.settings import MV_BASELINE_HL_DAYS

corr_rows = {}
for asset in ASSETS:
    actual  = er_test[asset]

    # EWMA naive forecast (5-year halflife, no look-ahead)
    ewma_fc = excess_returns[asset].ewm(
        halflife=MV_BASELINE_HL_DAYS, min_periods=252
    ).mean().shift(1).reindex(er_test.index)

    jmxgb_fc = regime_cond_returns[asset].reindex(er_test.index).shift(1)

    corr_ewma  = actual.corr(ewma_fc)
    corr_jmxgb = actual.corr(jmxgb_fc)
    corr_rows[asset] = {'EWMA': round(corr_ewma*100, 2), 'JM-XGB': round(corr_jmxgb*100, 2)}

table7 = pd.DataFrame(corr_rows).T
print('=== Table 7 — Forecast Correlation (%) ===')
print(table7.to_string())
table7.to_csv(f'{RESULTS_DIR}/table7_forecast_correlation.csv')

## 8 · Portfolio Backtesting (Table 6)

In [ ]:
from src.backtest.engine import BacktestEngine
from src.portfolio.strategies import build_strategy

engine = BacktestEngine(
    excess_returns   = excess_returns,
    total_returns    = total_returns,
    rf               = rf_daily,
    assets           = ASSETS,
)

strategy_names = [
    '60/40', 'MinVar', 'MinVar(JM-XGB)', 'MV', 'MV(JM-XGB)', 'EW', 'EW(JM-XGB)'
]

strategies = {name: build_strategy(name, ASSETS) for name in strategy_names}

PORT_CACHE = Path(f'{RESULTS_DIR}/portfolio_results.pkl')

if PORT_CACHE.exists():
    with open(PORT_CACHE, 'rb') as f:
        all_results = pickle.load(f)
else:
    all_results = engine.run_all(
        strategies        = strategies,
        test_start        = TEST_START,
        test_end          = TEST_END,
        regime_forecasts  = regime_forecasts,
        regime_cond_rets  = regime_cond_returns,
    )
    with open(PORT_CACHE, 'wb') as f:
        pickle.dump(all_results, f)

print('Backtesting complete.')

In [ ]:
from src.backtest.metrics import strategy_table

metrics_dict = {name: res['metrics'] for name, res in all_results.items()}
table6 = strategy_table(metrics_dict)

print('=== Table 6 — Portfolio Performance (2007-2023) ===')
print(table6.to_string())
table6.to_csv(f'{RESULTS_DIR}/table6_portfolio_performance.csv')

## 9 · Figure 3 — Strategy Wealth Curves

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 14))

groups = [
    (['MinVar(JM-XGB)', 'MinVar'], 'Strategy Performance Curves of Minimum-Variance Portfolios'),
    (['MV(JM-XGB)',  'MV', '60/40'], 'Strategy Performance Curves of Mean-Variance Portfolios'),
    (['EW(JM-XGB)', 'EW'],          'Strategy Performance Curves of Equally-Weighted Portfolios'),
]

palette = {'MinVar(JM-XGB)': 'steelblue', 'MinVar': 'darkorange',
           'MV(JM-XGB)':    'steelblue', 'MV':     'darkorange', '60/40': 'green',
           'EW(JM-XGB)':    'steelblue', 'EW':     'darkorange'}

for ax, (names, title) in zip(axes, groups):
    for name in names:
        if name in all_results:
            ret = all_results[name]['port_returns']
            wc  = wealth_curve(ret)
            ax.plot(wc.index, wc.values, label=name,
                    color=palette.get(name, 'grey'), lw=1.2)
    ax.set_yscale('log')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylabel('Wealth (log scale)')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig3_strategy_wealth_curves.png', dpi=150)
plt.show()

## 10 · Tables 8 & 9 — Sensitivity Analysis

In [ ]:
# Table 8 — MinVar(JM-XGB): γ_trade sensitivity
from src.portfolio.optimizer import MVOptimizer
from src.portfolio.strategies import MinVarJMXGB
from src.config.settings import GAMMA_RISK_MINVAR

sa_results = {}

for gamma_t in [0.0, 1.0]:
    class MinVarCustom(MinVarJMXGB):
        def __init__(self, assets):
            super().__init__(assets)
            self._opt = MVOptimizer(gamma_risk=GAMMA_RISK_MINVAR, gamma_trade=gamma_t)
    
    strat = MinVarCustom(ASSETS)
    res   = engine.run(strat, TEST_START, TEST_END,
                       regime_forecasts=regime_forecasts)
    sa_results[f'γ_trade={gamma_t}'] = res['metrics']

print('=== Table 8 — MinVar(JM-XGB) Trade Aversion Sensitivity ===')
print(strategy_table(sa_results).to_string())

In [ ]:
# Table 9 — MV(JM-XGB): γ_risk sensitivity
from src.portfolio.strategies import MVJMXGBPortfolio
from src.config.settings import GAMMA_TRADE_MV_JMXGB

sa9_results = {}

for gamma_r in [5.0, 10.0, 20.0]:
    class MVCustom(MVJMXGBPortfolio):
        def __init__(self, assets):
            super().__init__(assets)
            self._opt = MVOptimizer(gamma_risk=gamma_r, gamma_trade=GAMMA_TRADE_MV_JMXGB)
    
    strat = MVCustom(ASSETS)
    res   = engine.run(strat, TEST_START, TEST_END,
                       regime_forecasts=regime_forecasts,
                       regime_cond_rets=regime_cond_returns)
    sa9_results[f'γ_risk={gamma_r}'] = res['metrics']

print('=== Table 9 — MV(JM-XGB) Risk Aversion Sensitivity ===')
print(strategy_table(sa9_results).to_string())

## 11 · Statistical appendix — bootstrap Sharpe & trend / cycle in wealth

**Sharpe:** percentile bootstrap on i.i.d. resampling of **daily** `MV(JM-XGB)` portfolio returns (simple inferential summary; block bootstrap is preferable if one needs to preserve serial dependence).

**Trend / cycle / noise:** Hodrick–Prescott decomposition of **log wealth** ($\lambda = 1.6 \times 10^6$ for daily) yields a smooth trend and a cyclical residual; interpret as descriptive decomposition, not a structural model of returns.

In [ ]:
from src.analysis.time_series_stats import hp_decompose

def _annual_sharpe(r: pd.Series) -> float:
    m, s = float(r.mean()), float(r.std())
    return (np.sqrt(252) * m / s) if s > 1e-12 else 0.0

ret_mv = all_results["MV(JM-XGB)"]["port_returns"].dropna()
rng = np.random.default_rng(0)
B, n = 800, len(ret_mv)
boot_sr = []
for _ in range(B):
    idx = rng.integers(0, n, size=n)
    boot_sr.append(_annual_sharpe(ret_mv.iloc[idx]))
ci_lo, ci_hi = np.percentile(boot_sr, [2.5, 97.5])
print("MV(JM-XGB) annual Sharpe (point):", round(_annual_sharpe(ret_mv), 4))
print("Bootstrap 95% CI (i.i.d. resample):", round(ci_lo, 4), round(ci_hi, 4))

wc_mv = wealth_curve(ret_mv)
log_w = np.log(wc_mv.clip(lower=1e-8))
trend_w, cycle_w = hp_decompose(log_w, lam=1_600_000.0)
resid = log_w - trend_w
print("HP cycle std / residual std (log wealth):", float(cycle_w.std()), float(resid.std()))

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(log_w.index, log_w.values, label="log wealth", lw=1)
axes[0].plot(trend_w.index, trend_w.values, label="HP trend", lw=1.5)
axes[0].legend(fontsize=8)
axes[0].set_ylabel("log wealth")
axes[0].grid(alpha=0.3)
axes[1].plot(cycle_w.index, cycle_w.values, label="HP cycle", color="C2", lw=1)
axes[1].legend(fontsize=8)
axes[1].set_ylabel("cycle")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/fig_appendix_hp_logwealth_mv_jmxgb.png", dpi=150)
plt.show()